# Train engagement models (manual / Anaconda)

Use this notebook from **Anaconda Navigator** → **Jupyter Notebook** (or JupyterLab) to train the same pipeline as `train_engagement_models.py`.

### Before you start
1. In Navigator, pick an environment that has (or will get) **Python 3.10+**.
2. Launch **Jupyter Notebook**, browse to this folder:  
   `.../backEnd/microservices/Quiz/ml model`
3. Open **`train_engagement_manually.ipynb`**.

> **Working directory:** Jupyter often starts in `C:\Users\...` instead of the notebook folder. The next cell **auto-finds** `train_engagement_models.py` by searching from the current folder and common repo paths. If it still fails, set **`MANUAL_ML_ROOT`** in that cell to your full `ml model` path, or set env var **`ML_MODEL_ROOT`** before starting Jupyter.

### What you need in `dataset/`
- One or more **`.csv`** files.
- A column named **`Engagement_Level`** (target).
- Optional: drop **`Student_ID`** from features (handled automatically).

Training writes to **`output/<csv_stem>/`** (models + `statistics/*.png` for the admin **Expression insights** UI).

In [ ]:
# Optional: install packages into the active conda env (safe to re-run)
%pip install -q matplotlib numpy pandas scikit-learn joblib

In [ ]:
import os
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Optional: force the folder (use if auto-detect still fails). Otherwise leave as None.
# MANUAL_ML_ROOT = Path(r"D:\work\aymn2\...\backEnd\microservices\Quiz\ml model")
# ---------------------------------------------------------------------------
MANUAL_ML_ROOT = None  # type: ignore


def resolve_ml_model_dir() -> Path:
    env = os.environ.get("ML_MODEL_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "train_engagement_models.py").exists():
            return p

    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "train_engagement_models.py").exists():
            return p

    for rel in (
        Path("ml model"),
        Path("Quiz/ml model"),
        Path("microservices/Quiz/ml model"),
        Path("backEnd/microservices/Quiz/ml model"),
    ):
        cand = (here / rel).resolve()
        if (cand / "train_engagement_models.py").exists():
            return cand

    for p in [here, *here.parents]:
        for tail in (
            Path("backEnd/microservices/Quiz/ml model"),
            Path("microservices/Quiz/ml model"),
        ):
            cand = (p / tail).resolve()
            if (cand / "train_engagement_models.py").exists():
                return cand

    # Last resort: common clone path for this project (edit/remove if yours differs)
    _fallback = Path(
        r"D:\work\aymn2\application_web_distribu-es-UserModule-Final\application_web_distribu-es-UserModule-Final\backEnd\microservices\Quiz\ml model"
    )
    if (_fallback / "train_engagement_models.py").exists():
        return _fallback

    raise FileNotFoundError(
        "Could not find train_engagement_models.py.\n\n"
        "Jupyter often starts in C:\\Users\\... while the repo is on another drive.\n"
        "Fix one of:\n"
        "  1) Anaconda Prompt: cd \"...\\Quiz\\ml model\" then jupyter notebook\n"
        "  2) Set environment variable ML_MODEL_ROOT to your ml model folder\n"
        "  3) In this cell, replace MANUAL_ML_ROOT = None with Path(r\"D:\\...\\ml model\")"
    )


if MANUAL_ML_ROOT is not None:
    ML_ROOT = Path(MANUAL_ML_ROOT).expanduser().resolve()
else:
    ML_ROOT = resolve_ml_model_dir()

if not (ML_ROOT / "train_engagement_models.py").exists():
    raise FileNotFoundError(f"ML_ROOT does not contain train_engagement_models.py: {ML_ROOT}")

os.chdir(ML_ROOT)
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

print("ML_ROOT:", ML_ROOT)
print("CSV datasets:", sorted(p.name for p in (ML_ROOT / "dataset").glob("*.csv")))

### Run training
This **clears** the existing `output/` folder, then trains **every** `dataset/*.csv` that has `Engagement_Level`. It can take a few minutes.

In [ ]:
import train_engagement_models as trainer

trainer.main()
print("Done.")

### Preview PNG charts (facial track)
These files power **Expression insights** in the admin app when the dataset name is `student_engagement_with_facial_data`.

In [ ]:
from IPython.display import Image, display

facial_stats = ML_ROOT / "output" / "student_engagement_with_facial_data" / "statistics"
png_names = ["01_engagement_mix.png", "02_model_comparison.png", "03_test_confusion_matrix.png"]

if not facial_stats.is_dir():
    print("No facial statistics folder yet — check that dataset CSV trained successfully.")
else:
    for name in png_names:
        path = facial_stats / name
        if path.exists():
            print(name)
            display(Image(filename=str(path)))
        else:
            print("Missing:", name)

### Optional: inspect cross-validation table
Re-load one dataset and show the ranking without writing files (read-only peek).

In [ ]:
import pandas as pd
import train_engagement_models as trainer

csv_path = ML_ROOT / "dataset" / "student_engagement_with_facial_data.csv"
if not csv_path.exists():
    print("File not found:", csv_path.name)
else:
    df = pd.read_csv(csv_path)
    display(df.head())
    print("Shape:", df.shape, "| Target:", trainer.TARGET_COLUMN in df.columns)